# 🚀 Crypto Alpha Engine: High-Speed Colab Ingestion directly to Google Drive

This notebook downloads historical crypto market data (Klines 1h/15m, 8h Funding Rates, 5m Derivatives Metrics) directly from **Binance Data Vision (S3)** into **Google Drive** using Google's datacenter backbone (1 Gbps+).

* **Zero local disk space used on your Mac.**
* **Zero home internet bandwidth consumed.**
* **Zero idle disconnects**: Includes automatic keep-alive daemon.
* Stores clean, deduplicated, partitioned Parquet files (`zstd` compressed) in `MyDrive/trading/crypto-alpha-engine/data/`.

### 💡 Anti-Disconnect Keepalive (Optional for Browser Execution)
If you are executing via the browser instead of the headless CLI (`scripts/colab_runner.py`), run this in your browser console (F12 -> Console) to prevent idle timeouts:
```javascript
function ClickConnect(){ console.log("Keeping Colab alive..."); document.querySelector("colab-connect-button").click() }
setInterval(ClickConnect, 60000);
```

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_BASE_DIR = Path('/content/drive/MyDrive/trading/crypto-alpha-engine/data')
DRIVE_BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Target Storage Directory: {DRIVE_BASE_DIR}")

In [ ]:
# 2. Clone Repository & Install High-Performance Cloud Dependencies
!git clone https://github.com/sanjmen/crypto-alpha-engine.git /content/crypto-alpha-engine || (cd /content/crypto-alpha-engine && git pull)
%cd /content/crypto-alpha-engine
!pip install duckdb pyarrow pandas requests --quiet

In [ ]:
# 3. Execute High-Throughput Ingestion Worker with Thread Pool
!python scripts/colab_ingest_worker.py

In [ ]:
# 4. Fast Verification via DuckDB on Google Drive Parquet Storage
import duckdb

btc_1h = f"{DRIVE_BASE_DIR}/bars/1h/BTCUSDT.parquet"
btc_funding = f"{DRIVE_BASE_DIR}/funding/BTCUSDT.parquet"

if Path(btc_1h).exists():
    print("\n--- BTC/USDT 1h Bars Sample ---")
    df_bars = duckdb.query(f"SELECT to_timestamp(timestamp) as time, open, high, low, close, volume FROM read_parquet('{btc_1h}') ORDER BY timestamp DESC LIMIT 5").to_df()
    display(df_bars)

if Path(btc_funding).exists():
    print("\n--- BTC/USDT Funding Rate Sample ---")
    df_fund = duckdb.query(f"SELECT to_timestamp(timestamp) as time, funding_rate, interval_hours FROM read_parquet('{btc_funding}') ORDER BY timestamp DESC LIMIT 5").to_df()
    display(df_fund)